# 03 — Inventory Optimization

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

demand = pd.read_csv("../data/raw/weekly_demand.csv", parse_dates=["week_start"])
sku = pd.read_csv("../data/raw/sku_master.csv")
profile = pd.read_csv("../data/processed/sku_demand_profile.csv")

In [ ]:
stats = demand.groupby("sku_id")["demand_units"].agg(
    avg_weekly_demand="mean", demand_std="std"
).reset_index()

inv = sku.merge(stats,on="sku_id").merge(
    profile[["sku_id","abc_class","xyz_class"]], on="sku_id"
)

inv["annual_demand"] = inv["avg_weekly_demand"]*52
inv["annual_holding_cost_per_unit"] = inv["unit_cost"]*inv["annual_holding_rate"]
inv["eoq_units"] = np.sqrt(
    (2*inv["annual_demand"]*inv["ordering_cost"]) /
    inv["annual_holding_cost_per_unit"]
)
inv["z_score"] = inv["target_service_level"].apply(norm.ppf)
inv["safety_stock_units"] = inv["z_score"]*inv["demand_std"]*np.sqrt(inv["lead_time_weeks"])
inv["reorder_point_units"] = inv["avg_weekly_demand"]*inv["lead_time_weeks"] + inv["safety_stock_units"]
inv["annual_ordering_cost"] = (inv["annual_demand"]/inv["eoq_units"])*inv["ordering_cost"]
inv["annual_cycle_stock_holding_cost"] = (inv["eoq_units"]/2)*inv["annual_holding_cost_per_unit"]
inv["annual_safety_stock_holding_cost"] = inv["safety_stock_units"]*inv["annual_holding_cost_per_unit"]
inv["estimated_annual_inventory_cost"] = inv[
    ["annual_ordering_cost","annual_cycle_stock_holding_cost","annual_safety_stock_holding_cost"]
].sum(axis=1)

for c in ["eoq_units","safety_stock_units","reorder_point_units"]:
    inv[c] = inv[c].round(0).astype(int)

inv.to_csv("../data/processed/inventory_policy.csv", index=False)
display(inv[[
    "sku_id","sku_name","abc_class","xyz_class","lead_time_weeks",
    "target_service_level","eoq_units","safety_stock_units",
    "reorder_point_units","estimated_annual_inventory_cost"
]])